In [1]:
import pandas as pd
# %pip install pyarrow
# %pip install fastparquet
%pip install openpyxl

import openpyxl


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
df = pd.read_csv(r"/Users/kamlesh/Desktop/Projects/IKEA CustomerAnalysis/data/RetailCleaned.csv")

/var/folders/12/5lts56994mb9br219_njbqww0000gn/T/ipykernel_6788/1919260992.py:1: DtypeWarning: Columns (0: invoice) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"/Users/kamlesh/Desktop/Projects/IKEA CustomerAnalysis/data/RetailCleaned.csv")


In [3]:
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
df['customer_id'] = df['customer_id'].astype('Int64')

In [4]:
print(df.shape)

(1033036, 9)


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   invoice      1033036 non-null  object        
 1   stockcode    1033036 non-null  str           
 2   description  1028761 non-null  str           
 3   quantity     1033036 non-null  int64         
 4   invoicedate  1033036 non-null  datetime64[us]
 5   price        1033036 non-null  float64       
 6   customer_id  797885 non-null   Int64         
 7   country      1033036 non-null  str           
 8   sourcesheet  1033036 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), object(1), str(4)
memory usage: 125.3+ MB


### FILTER AND CALCULATE RECENCY, FREQUENCY, AND MONETARY

In [6]:
rfm_data = df[(df['customer_id'].notna()) & (df['quantity']>0)].copy()

#SNAPSHOT IS THE DATE ON WHICH THE ANALYSIS IS BEING PERFORMED
snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('customer_id').agg(
    recency = ('invoicedate',lambda x:(snapshot_date-x.max()).days),
    frequency = ('invoice','nunique'),
    monetary = ('quantity',lambda x:(x * rfm_data.loc[x.index,'price']).sum())
).reset_index()

In [7]:
#SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS
rfm['R_Score'] = pd.qcut(rfm['recency'],5,labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['frequency'].rank(method='first'),5,labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['monetary'],5,labels=[1,2,3,4,5]).astype(int)

In [8]:
def segment_customer(row):
    if row['R_Score']>=4 and row['F_Score']>=4 and row['M_Score']>=4:
        return 'Champions'
    elif row['R_Score']>=3 and row['F_Score']>=3:
        return 'Loyal'
    elif row['R_Score']<=2 and row['F_Score']>=3:
        return 'At Risk'
    elif row['R_Score']>=4 and row['F_Score']<=2:
        return 'New'
    elif row['R_Score']==3 and row['F_Score']<=2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer,axis=1)
rfm['isChurned'] = rfm['recency']>90   

In [9]:
# MERGE RFM RESULTS BACK INTO TRANSACTIONS AND EXPORT
df_final = df.merge(rfm[['customer_id', 'recency', 'frequency', 'monetary', 'R_Score', 'F_Score',
       'M_Score', 'Segment', 'isChurned']],on = 'customer_id',how='left')

df_final

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,sourcesheet,recency,frequency,monetary,R_Score,F_Score,M_Score,Segment,isChurned
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-2010,157.0,8.0,2433.28,3.0,4.0,4.0,Loyal,True
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085,United Kingdom,2009-2010,157.0,8.0,2433.28,3.0,4.0,4.0,Loyal,True
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-2010,157.0,8.0,2433.28,3.0,4.0,4.0,Loyal,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009-2010,157.0,8.0,2433.28,3.0,4.0,4.0,Loyal,True
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom,2009-2010,157.0,8.0,2433.28,3.0,4.0,4.0,Loyal,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1033031,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397,United Kingdom,2010-2011,2.0,19.0,2612.96,5.0,5.0,4.0,Champions,False
1033032,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498,United Kingdom,2010-2011,2.0,44.0,24972.86,5.0,5.0,5.0,Champions,False
1033033,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311,United Kingdom,2010-2011,0.0,208.0,114966.42,5.0,5.0,5.0,Champions,False
1033034,C581569,20979,36 PENCILS TUBE RED RETROSPOT,-5,2011-12-09 11:58:00,1.25,17315,United Kingdom,2010-2011,1.0,43.0,9599.58,5.0,5.0,5.0,Champions,False


In [10]:
df_final.to_excel(r"/Users/kamlesh/Desktop/Projects/IKEA CustomerAnalysis/data/EcommerceDashboardData.xlsx",index=False)

In [11]:
df_final.dtypes

invoice                object
stockcode                 str
description               str
quantity                int64
invoicedate    datetime64[us]
price                 float64
customer_id             Int64
country                   str
sourcesheet               str
recency               float64
frequency             float64
monetary              float64
R_Score               float64
F_Score               float64
M_Score               float64
Segment                   str
isChurned              object
dtype: object